# 08 · Mid-session electrode changes

When each channel died / revived.  A channel is "dead in a 10-s bin" if >95% of consecutive
samples are identical (a pinned rail).  Supports the report's §2.1 dynamic-channel notes.

In [1]:
import warnings, pathlib, sys
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
_c = pathlib.Path.cwd()
REPO = next(p for p in [_c] + list(_c.parents) if (p / "analysis").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from analysis import dataset, eegkit as K
DATA_DIR = None
recs = dataset.discover(DATA_DIR)
_gi, LABEL = {}, {}
for _r in recs:
    _g = _r["group"]; _gi[_g] = _gi.get(_g, 0) + 1
    LABEL[_r["key"]] = ("exp" if _g == "E" else "ctrl") + str(_gi[_g])
def lbl(r): return LABEL[r["key"]]
FIG = REPO / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def ec_window(r):
    st = K.get_stages(K.load(r["path"]), r["key"])
    return [s for s in st if s[0].startswith("rest_close")][0]
def get_iaf(r):
    raw = K.load(r["path"]); ec = ec_window(r)
    _, live = K.get(raw, ec[1], ec[2]); return K.iaf(raw, live, ec[1], ec[2])
def stage_of(r, nm):
    st = K.get_stages(K.load(r["path"]), r["key"]); return [s for s in st if s[0] == nm]
print(f"discovered {len(recs)} recordings ->", {lbl(r): r["group"] for r in recs})


discovered 4 recordings -> {'exp1': 'E', 'exp2': 'E', 'ctrl1': 'C', 'ctrl2': 'C'}


In [2]:
BIN=10
for r in recs:
    raw=K.load(r["path"]); fs=raw.info["sfreq"]; x=raw.get_data()
    dur=raw.n_times/fs; nb=int(dur//BIN)
    print("="*70); print(lbl(r))
    for name in K.eeg_channels(raw):
        d=x[raw.ch_names.index(name)]*1e6
        flat=np.array([(np.diff(d[int(b*BIN*fs):int((b+1)*BIN*fs)])==0).mean()>0.95 for b in range(nb)])
        if not flat.any(): continue
        ch=np.diff(flat.astype(int)); starts=list(np.where(ch==1)[0]+1); ends=list(np.where(ch==-1)[0]+1)
        if flat[0] and not ends: desc="DEAD whole recording"
        elif flat[0] and ends:   desc=f"dead 0-{ends[0]*BIN}s, revived@{ends[0]*BIN}s"
        elif starts and not ends: desc=f"OK until died@{starts[0]*BIN}s"
        elif starts and ends:    desc=f"OK until died@{starts[0]*BIN}s, revived@{ends[0]*BIN}s"
        else:                     desc="flapping"
        print(f"  {name:<6} {desc:<50} ({flat.mean()*100:.0f}% flat)")
print("\nSame dead ring every session; FC3/POz/CP4 change state mid-session -> time-aware bad lists.")


exp1
  Fpz    DEAD whole recording                               (100% flat)
  FT7    DEAD whole recording                               (100% flat)
  FC3    dead 0-1050s, revived@1050s                        (54% flat)
  FCz    DEAD whole recording                               (100% flat)
  FT8    DEAD whole recording                               (100% flat)
  TP7    DEAD whole recording                               (100% flat)
  CP3    DEAD whole recording                               (100% flat)
  CPz    DEAD whole recording                               (100% flat)
  TP8    DEAD whole recording                               (100% flat)
  P5     DEAD whole recording                               (100% flat)
  POz    OK until died@90s                                  (96% flat)
  P6     DEAD whole recording                               (100% flat)


exp2
  Fpz    DEAD whole recording                               (100% flat)
  FT7    DEAD whole recording                               (100% flat)
  FC3    DEAD whole recording                               (100% flat)
  FCz    DEAD whole recording                               (100% flat)
  FC4    DEAD whole recording                               (100% flat)
  FT8    DEAD whole recording                               (100% flat)
  CP3    DEAD whole recording                               (100% flat)
  CPz    DEAD whole recording                               (100% flat)
  CP4    OK until died@1250s                                (34% flat)
  TP8    DEAD whole recording                               (100% flat)
  POz    DEAD whole recording                               (100% flat)
  P6     DEAD whole recording                               (100% flat)


ctrl1
  Fp1    OK until died@1390s                                (1% flat)
  Fp2    OK until died@1390s                                (1% flat)
  Fpz    dead 0-1380s, revived@1380s                        (99% flat)
  F7     OK until died@1390s                                (1% flat)
  F3     OK until died@1390s                                (1% flat)
  Fz     OK until died@1390s                                (1% flat)
  F4     OK until died@1390s                                (1% flat)
  FT7    dead 0-1380s, revived@1380s                        (99% flat)
  FCz    dead 0-1380s, revived@1380s                        (99% flat)
  FC4    dead 0-1380s, revived@1380s                        (99% flat)
  FT8    dead 0-1380s, revived@1380s                        (99% flat)
  TP7    dead 0-1380s, revived@1380s                        (99% flat)
  CP3    dead 0-1380s, revived@1380s                        (99% flat)
  CPz    dead 0-1380s, revived@1380s                        (99% flat)
  CP4 

ctrl2
  Fpz    DEAD whole recording                               (100% flat)
  FT7    DEAD whole recording                               (100% flat)
  FCz    DEAD whole recording                               (100% flat)
  FC4    DEAD whole recording                               (100% flat)
  FT8    DEAD whole recording                               (100% flat)
  TP7    DEAD whole recording                               (100% flat)
  CP3    DEAD whole recording                               (100% flat)
  CPz    DEAD whole recording                               (100% flat)
  CP4    DEAD whole recording                               (100% flat)
  TP8    DEAD whole recording                               (100% flat)
  P5     DEAD whole recording                               (100% flat)
  POz    DEAD whole recording                               (100% flat)
  P6     DEAD whole recording                               (100% flat)

Same dead ring every session; FC3/POz/CP4 change state mi